In [54]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import os

In [55]:
paths = [
    "../data/processed/split/X_train.csv",
    "../data/processed/split/X_test.csv"
]

X_train = pd.read_csv(paths[0])
X_test  = pd.read_csv(paths[1])

In [56]:
exames_cols = X_train.select_dtypes(include=["float64"]).columns.tolist()

cols_para_log = [
    col
    for col in exames_cols
    if X_train[col].skew() > 1 and (X_train[col] >= 0).all()
]

print(
    f"Aplicando log1p em {len(cols_para_log)} de {len(exames_cols)} exames:\n{cols_para_log}"
)

X_train_prep = X_train.copy()
X_test_prep = X_test.copy()

X_train_prep[cols_para_log] = np.log1p(X_train[cols_para_log])
X_test_prep[cols_para_log] = np.log1p(X_test[cols_para_log])

Aplicando log1p em 14 de 23 exames:
['WBC', 'NE#', 'LY#', 'MO#', 'EO#', 'BA#', 'RDW', 'PLT', 'PCT', 'SD', 'TSD', 'FERRITTE', 'FOLATE', 'B12']


In [57]:
num_cols = X_train_prep.select_dtypes(include=["float64"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[("num", StandardScaler(), num_cols)],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X_train_scaled = preprocessor.fit_transform(X_train_prep)
X_test_scaled = preprocessor.transform(X_test_prep)

In [58]:
display(X_train_scaled.head())

,WBC,NE#,LY#,MO#,EO#,BA#,RBC,HGB,HCT,MCV,...,MPV,PCT,PDW,SD,SDTSD,TSD,FERRITTE,FOLATE,B12,GENDER
0,-0.632472,-0.527215,-0.256992,-0.136378,-0.808066,-0.343198,0.391721,-0.582208,-0.111164,-0.785838,...,0.641086,0.246326,0.558151,-1.140014,1.004998,-1.421261,-1.167369,0.686149,-0.773794,0
1,-0.841033,-0.686936,-0.381983,-0.798582,-0.493951,-0.027988,0.195133,0.973355,0.413967,0.289030,...,-0.569887,-0.386244,0.389257,0.747713,-0.416015,0.801648,0.062452,0.919380,0.463783,0
2,-0.668557,-0.705299,-0.256992,0.300280,0.100807,-0.343198,-0.061943,0.537797,0.421136,0.714498,...,1.050350,-0.257630,-1.307028,-0.332903,-0.279493,-0.272619,-0.168695,2.266226,-0.074646,0
3,-0.170201,-0.355078,0.462504,-0.506912,0.722736,-0.027988,0.285866,0.920705,0.833355,0.780360,...,0.489714,0.369775,-1.502847,0.853174,-0.315521,0.861204,0.170007,-0.307591,-1.409032,1
4,0.984879,0.729769,0.991455,1.946968,0.382972,-0.664528,-0.772682,-0.275882,-0.851366,-0.207569,...,0.657905,-0.130083,-1.943441,0.376732,-0.604953,0.528395,1.132847,0.279713,1.523601,0


In [59]:
def plot_distributions(col):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    is_discrete = X_train[col].nunique() <= 10
    usou_log = col in cols_para_log
    foi_escalonado = col in num_cols

    sns.histplot(
        X_train[col],
        color="#1f77b4",
        label="Treino",
        kde=not is_discrete,
        stat="density",
        common_norm=False,
        discrete=is_discrete,
        bins=30,
        alpha=0.4,
        ax=axes[0],
    )
    sns.histplot(
        X_test[col],
        color="#ff7f0e",
        label="Teste",
        kde=not is_discrete,
        stat="density",
        common_norm=False,
        discrete=is_discrete,
        bins=30,
        alpha=0.4,
        ax=axes[0],
    )
    axes[0].set_title(f"Antes do Escalonamento: {col}", fontsize=13)
    axes[0].set_xlabel("Valor Original")
    axes[0].set_ylabel("Densidade")
    axes[0].legend()

    sns.histplot(
        X_train_scaled[col],
        color="#1f77b4",
        label="Treino (Processado)",
        kde=not is_discrete,
        stat="density",
        common_norm=False,
        discrete=is_discrete,
        bins=30,
        alpha=0.4,
        ax=axes[1],
    )
    sns.histplot(
        X_test_scaled[col],
        color="#ff7f0e",
        label="Teste (Processado)",
        kde=not is_discrete,
        stat="density",
        common_norm=False,
        discrete=is_discrete,
        bins=30,
        alpha=0.4,
        ax=axes[1],
    )

    if usou_log:
        titulo_dir = f"Depois do Log + StandardScaler: {col}"
        xlabel_dir = "Valor Escalonado (log + Z-score)"
    elif foi_escalonado:
        titulo_dir = f"Depois do StandardScaler: {col}"
        xlabel_dir = "Valor Escalonado (Z-score)"
    else:
        titulo_dir = f"Sem Escalonamento (Passthrough): {col}"
        xlabel_dir = "Valor Original"

    axes[1].set_title(titulo_dir, fontsize=13)
    axes[1].set_xlabel(xlabel_dir)
    axes[1].set_ylabel("Densidade")
    axes[1].legend()

    if is_discrete:
        valores_unicos = sorted(X_train[col].unique())
        axes[0].set_xticks(valores_unicos)
        axes[1].set_xticks(valores_unicos)

    plt.tight_layout()
    plt.show()

In [60]:
seletor = widgets.Dropdown(options=X_train.columns.tolist(), description="Coluna:")
out = widgets.interactive_output(plot_distributions, {"col": seletor})
display(seletor, out)

Dropdown(description='Coluna:', options=('GENDER', 'WBC', 'NE#', 'LY#', 'MO#', 'EO#', 'BA#', 'RBC', 'HGB', 'HC…

Output()

In [61]:
comparacao = []

for col in X_train.columns:
    if col in cols_para_log:
        tratamento = "Log1p + StandardScaler"
    elif col in num_cols:
        tratamento = "StandardScaler"
    else:
        tratamento = "Nenhum (Passthrough)"

    comparacao.append({
        "Coluna": col,
        "Tratamento": tratamento,
        
        # Escala Original
        "Min (Antes)": round(X_train[col].min(), 2),
        "Mediana (Antes)": round(X_train[col].median(), 2),
        "Max (Antes)": round(X_train[col].max(), 2),
        "Assimetria (Antes)": round(X_train[col].skew(), 2),
        
        # Escala Transformada
        "Min (Depois)": round(X_train_scaled[col].min(), 2),
        "Mediana (Depois)": round(X_train_scaled[col].median(), 2),
        "Max (Depois)": round(X_train_scaled[col].max(), 2),
        "Assimetria (Depois)": round(X_train_scaled[col].skew(), 2),
    })

df_comparacao = pd.DataFrame(comparacao)

with pd.option_context("display.max_rows", None):
    display(df_comparacao)


,Coluna,Tratamento,Min (Antes),Mediana (Antes),Max (Antes),Assimetria (Antes),Min (Depois),Mediana (Depois),Max (Depois),Assimetria (Depois)
0,GENDER,Nenhum (Passthrough),0.00,0.00,1.00,0.77,0.00,0.00,1.00,0.77
1,WBC,Log1p + StandardScaler,0.57,7.51,246.70,19.45,-5.37,-0.03,10.61,0.61
2,NE#,Log1p + StandardScaler,0.03,4.42,197.20,19.22,-4.44,-0.08,9.36,0.59
3,LY#,Log1p + StandardScaler,0.11,2.13,89.60,24.53,-3.60,0.05,11.91,1.10
4,MO#,Log1p + StandardScaler,0.01,0.52,18.51,22.08,-2.94,-0.13,17.35,2.03
5,EO#,Log1p + StandardScaler,0.00,0.12,9.41,18.35,-1.13,-0.21,18.36,3.76
6,BA#,Log1p + StandardScaler,0.00,0.05,24.73,85.70,-0.99,-0.12,52.75,22.28
7,RBC,StandardScaler,1.20,4.70,7.36,-0.21,-5.28,0.01,4.04,-0.21
8,HGB,StandardScaler,3.79,12.83,22.45,-0.23,-4.30,0.03,4.63,-0.23
9,HCT,StandardScaler,13.67,39.15,61.90,-0.24,-4.53,0.04,4.11,-0.24


In [62]:
PATH = "../data/processed/scaled"

if not os.path.exists(PATH):
    os.makedirs(PATH)

files = {
    "../data/processed/scaled/X_train_scaled.csv": X_train_scaled,
    "../data/processed/scaled/X_test_scaled.csv": X_test_scaled,
}

for file_path, data in files.items():
    data.to_csv(file_path, index=False)
    print(f"Salvo: {file_path}")

Salvo: ../data/processed/scaled/X_train_scaled.csv
Salvo: ../data/processed/scaled/X_test_scaled.csv
